# Import data

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go


## Load the CSV file into a DataFrame

The file has headers in the first row. `pandas.read_csv()` reads them automatically and creates a `DataFrame` from the data.

In [12]:


csv_candidates = (
    Path.cwd() / "src" / "metal_bands.csv",
    Path.cwd().parent / "src" / "metal_bands.csv",
)
csv_path = next((path for path in csv_candidates if path.is_file()), None)

if csv_path is None:
    raise FileNotFoundError("Could not find src/metal_bands.csv.")

df = pd.read_csv(csv_path, dtype={"Band ID": "string"})

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
print("Headers:", df.columns.tolist())
display(df.head())

Loaded 183397 rows and 7 columns.
Headers: ['Band ID', 'Name', 'URL', 'Country', 'Genre', 'Status', 'Photo_URL']


,Band ID,Name,URL,Country,Genre,Status,Photo_URL
0,3540442600,A // Solution,https://www.metal-archives.com/bands/A_--_Solu...,United States,Crust Punk/Thrash Metal,Split-up,NaN
1,3540525193,A Black Cold Diamond,https://www.metal-archives.com/bands/A_Black_C...,Colombia,Heavy/Doom Metal,Active,https://www.metal-archives.com/images/3/5/4/0/...
2,3540473101,A Billion Limbs,https://www.metal-archives.com/bands/A_Billion...,United Kingdom,Groove Metal/Deathcore,Active,NaN
3,3540535978,A Band Named Jon,https://www.metal-archives.com/bands/A_Band_Na...,United States,Brutal Death Metal/Grindcore,Active,https://www.metal-archives.com/images/3/5/4/0/...
4,3540352307,A Balance of Power,https://www.metal-archives.com/bands/A_Balance...,United States,Melodic Death Metal/Metalcore,Active,https://www.metal-archives.com/images/3/5/4/0/...


#ADDING COUNTER

In [13]:
df["COUNTER"]=1


#HOW MANY BANDS PER COUNTRY 

In [14]:
COUNTRY_BANDS=df[["Country","COUNTER"]].groupby("Country").sum().sort_values(by="COUNTER",ascending=False)

#RESULT

#ONLY EUROPE


In [15]:
EUROPEAN_COUNTRIES = {
    "Albania", "Andorra", "Armenia", "Austria", "Azerbaijan", "Belarus",
    "Belgium", "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus",
    "Czechia", "Denmark", "Estonia", "Finland", "France", "Georgia",
    "Germany", "Greece", "Hungary", "Iceland", "Ireland", "Italy",
    "Kazakhstan", "Latvia", "Liechtenstein", "Lithuania", "Luxembourg",
    "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands",
    "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Russia",
    "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Switzerland", "T\u00fcrkiye", "Ukraine", "United Kingdom", "Vatican City",
}
COUNTRY_BANDS_EUROPE = COUNTRY_BANDS[COUNTRY_BANDS.index.isin(EUROPEAN_COUNTRIES)]

In [16]:
print(COUNTRY_BANDS_EUROPE)

                        COUNTER
Country                        
Germany                   13840
Italy                      8100
United Kingdom             7513
France                     6749
Russia                     6237
Sweden                     5966
Finland                    5469
Spain                      5245
Poland                     4429
Netherlands                3090
Greece                     2665
Norway                     2396
Czechia                    1881
Portugal                   1686
Belgium                    1673
Austria                    1585
Ukraine                    1535
Hungary                    1447
Switzerland                1383
Denmark                    1331
Slovakia                    753
Türkiye                     742
Croatia                     609
Romania                     581
Ireland                     571
Serbia                      570
Belarus                     556
Bulgaria                    548
Slovenia                    345
Estonia 

In [17]:
country_centroids = {
    "Albania": (41.2, 20.2), "Andorra": (42.5, 1.5),
    "Austria": (47.5, 14.5), "Belarus": (53.5, 28.0),
    "Belgium": (50.5, 4.5), "Bosnia and Herzegovina": (44.2, 17.7),
    "Bulgaria": (42.7, 25.5), "Croatia": (45.1, 15.2),
    "Czechia": (49.8, 15.5),
    "Denmark": (56.0, 9.5), "Estonia": (58.7, 25.0),
    "Finland": (64.0, 26.0), "France": (46.2, 2.2),
    "Germany": (51.2, 10.4),
    "Greece": (39.0, 22.0), "Hungary": (47.2, 19.5),
    "Iceland": (64.9, -18.6), "Ireland": (53.4, -8.0),
    "Italy": (42.8, 12.6),
    "Latvia": (56.9, 24.6), "Liechtenstein": (47.2, 9.6),
    "Lithuania": (55.2, 23.9), "Luxembourg": (49.8, 6.1),
    "Malta": (35.9, 14.4), "Moldova": (47.0, 28.9),
    "Monaco": (43.7, 7.4), "Montenegro": (42.4, 19.3),
    "Netherlands": (52.1, 5.3), "North Macedonia": (41.6, 21.7),
    "Norway": (61.0, 8.0), "Poland": (52.1, 19.1),
    "Portugal": (39.6, -8.0), "Romania": (45.9, 24.9),
    "Russia": (56.0, 38.0), "San Marino": (43.9, 12.5),
    "Serbia": (44.0, 20.8), "Slovakia": (48.7, 19.5),
    "Slovenia": (46.1, 14.8), "Spain": (40.2, -3.7),
    "Sweden": (60.1, 18.6), "Switzerland": (46.8, 8.2),
    "Ukraine": (49.0, 32.0),
    "United Kingdom": (54.5, -2.5), "Vatican City": (41.9, 12.5),
}

map_data = COUNTRY_BANDS_EUROPE.reset_index().rename(columns={"Country": "country"})
unmapped_countries = sorted(set(map_data["country"]) - set(country_centroids))
if unmapped_countries:
    print("Countries without geographic centroids (omitted):", ", ".join(unmapped_countries))
map_data = map_data[map_data["country"].isin(country_centroids)].copy()
map_data[["latitude", "longitude"]] = map_data["country"].map(country_centroids).apply(pd.Series)

counts = map_data["COUNTER"].astype(float)
sqrt_counts = np.sqrt(counts)
count_span = sqrt_counts.max() - sqrt_counts.min()
skull_sizes = 14 + 34 * (sqrt_counts - sqrt_counts.min()) / count_span if count_span else 24
skull = "\U0001F480"

fig = go.Figure(go.Scattergeo(
    lat=map_data["latitude"],
    lon=map_data["longitude"],
    mode="text",
    text=[skull] * len(map_data),
    textfont=dict(size=skull_sizes, family="Segoe UI Emoji, Noto Color Emoji, sans-serif"),
    customdata=map_data[["country", "COUNTER"]].to_numpy(),
    hovertemplate="<b>%{customdata[0]}</b><br>Bands: %{customdata[1]}<extra></extra>",
    textposition="middle center",
    showlegend=False,
))

fig.update_layout(
    title="Metal bands in Europe (larger skull = more bands)",
    template="plotly_dark",
    paper_bgcolor="rgb(13, 20, 30)",
    plot_bgcolor="rgb(13, 20, 30)",
    height=650,
    geo=dict(
        scope="europe",
        projection_type="natural earth",
        showframe=False,
        showland=True,
        landcolor="rgb(46, 58, 70)",
        showocean=True,
        oceancolor="rgb(13, 20, 30)",
        showcountries=True,
        countrycolor="rgb(105, 125, 145)",
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()


Countries without geographic centroids (omitted): Armenia, Azerbaijan, Cyprus, Georgia, Kazakhstan, Türkiye
